# Data Cleaning:

In [8]:
import pandas as pd
import os
from PIL import Image

from sklearn.model_selection import train_test_split


### Step-1: Load the dataset:

In [9]:
df = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\dataset.csv")

### Create a new cleaned dataset csv file where image exists:

In [10]:
df['image_exists'] = df['image_id'].apply(
    lambda x: os.path.exists(
        f"C:/Users/jagad/Documents/my_classes/Tasks/my_projects/Deep_Fake_project_final/data/raw_images/{x}.jpg"
                            ))

cleaned_df = df[df['image_exists'] == True].copy()

print("Original rows:", len(df))
print("After removing failed downloads:", len(cleaned_df))

cleaned_df.to_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\cleaned_dataset.csv", index=False)

print("Saved as data/cleaned_dataset.csv")

Original rows: 6557
After removing failed downloads: 5557
Saved as data/cleaned_dataset.csv


# Image Verification:

In [11]:
# Read the clean dataset
df = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\cleaned_dataset.csv")

corrupted_ids = []

for image_id in df['image_id']:
    file_path = f"C:/Users/jagad/Documents/my_classes/Tasks/my_projects/Deep_Fake_project_final/data/raw_images/{image_id}.jpg"
    try:
        img = Image.open(file_path)
        img.verify()   # checks if the image file is valid, without fully loading it
    except Exception as e:
        corrupted_ids.append(image_id)


print("Total images checked:", len(df))
print("Corrupted images found:", len(corrupted_ids))


# Remove corrupted rows and save final clean dataset
final_df = df[~df['image_id'].isin(corrupted_ids)]
final_df.to_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\cleaned_dataset.csv", index=False)

print("Saved as data/final_dataset.csv with", len(final_df), "rows")

Total images checked: 5557
Corrupted images found: 0
Saved as data/final_dataset.csv with 5557 rows


In [12]:
import hashlib
from pathlib import Path

DATA_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data")
IMG_DIR = DATA_DIR / "raw_images"

final_df = pd.read_csv(DATA_DIR / "final_dataset.csv")

# 1) same URL listed more than once
print("Duplicate URLs:", final_df["image_url"].duplicated().sum())

# 2) same image content (exact copy), even if the ID or URL is different
def file_md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

final_df["md5"] = final_df["image_id"].apply(lambda i: file_md5(IMG_DIR / f"{i}.jpg"))
dup_mask = final_df["md5"].duplicated(keep=False)
print("Images involved in exact duplicates:", dup_mask.sum())

# same picture labelled both REAL and FAKE? (check these by hand if > 0)
conflict = final_df[dup_mask].groupby("md5")["label"].nunique()
print("Duplicate groups with conflicting labels:", (conflict > 1).sum())

# keep only the first copy of each image
final_df = final_df.drop_duplicates(subset="md5", keep="first").drop(columns=["md5"])
print("Rows after removing duplicates:", len(final_df))

final_df.to_csv(DATA_DIR / "final_dataset.csv", index=False)

Duplicate URLs: 0
Images involved in exact duplicates: 0
Duplicate groups with conflicting labels: 0
Rows after removing duplicates: 5114


# Data Splitting:

### Train/Test/Val Split

In [13]:
df1 = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\final_dataset.csv")
img_dirs = r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\raw_images"

print("First five rows:")
print(df1.head())
print("-" * 50)

print("Datashape: ", df1.shape)
print("-" * 50)

First five rows:
   image_id                                          image_url label  gender  \
0         1  https://images.unsplash.com/photo-172422561835...  REAL  Female   
1         2  https://images.unsplash.com/photo-172422561873...  REAL    Male   
2         3  https://images.unsplash.com/photo-172422561834...  REAL  Female   
3         4  https://images.unsplash.com/photo-172422561855...  REAL  Female   
4         5  https://images.unsplash.com/photo-172422561823...  REAL    Male   

  age_group  image_exists  
0     18-25          True  
1       50+          True  
2     36-50          True  
3     26-35          True  
4     36-50          True  
--------------------------------------------------
Datashape:  (5114, 6)
--------------------------------------------------


### Test Splitting:

In [14]:
train_val_df, test_df = train_test_split(   df1,
                                            test_size= 0.15 ,        
                                            stratify=df1['label'],   
                                            random_state=42
                                        )

print("Train and Val size:", train_val_df.shape)
print("Test size:", test_df.shape)

Train and Val size: (4346, 6)
Test size: (768, 6)


### Train and Val splitting:

In [15]:
train_df, val_df = train_test_split(    train_val_df,                         
                                        test_size= 0.176,             
                                        stratify= train_val_df['label'],     
                                        random_state=42
                                    )

print("Train size:", train_df.shape)
print("Val size:", val_df.shape)

Train size: (3581, 6)
Val size: (765, 6)


In [16]:
print("Train label distribution:")
print(train_df['label'].value_counts(normalize=True))

print("\nValidation label distribution:")
print(val_df['label'].value_counts(normalize=True))

print("\nTest label distribution:")
print(test_df['label'].value_counts(normalize=True))

Train label distribution:
label
FAKE    0.537001
REAL    0.462999
Name: proportion, dtype: float64

Validation label distribution:
label
FAKE    0.537255
REAL    0.462745
Name: proportion, dtype: float64

Test label distribution:
label
FAKE    0.536458
REAL    0.463542
Name: proportion, dtype: float64


### Saving three splits into complete separate csv file:

In [17]:
train_df.to_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\train.csv", index=False)
val_df.to_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\val.csv", index=False)
test_df.to_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\test.csv", index=False)

print("Saved train.csv, val.csv, test.csv")

Saved train.csv, val.csv, test.csv


In [18]:
import shutil
from pathlib import Path

DATA_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data")
SRC = DATA_DIR / "raw_images"
OUT = DATA_DIR / "split_images"

# Start fresh so files from an older split are not left behind
shutil.rmtree(OUT, ignore_errors=True)

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    dest = OUT / split_name
    dest.mkdir(parents=True, exist_ok=True)
    for image_id in split_df["image_id"]:
        shutil.copy2(SRC / f"{image_id}.jpg", dest / f"{image_id}.jpg")
    print(split_name, "->", len(list(dest.glob("*.jpg"))), "images")

# Sanity checks: no image in two splits, and nothing lost
ids = [set(train_df["image_id"]), set(val_df["image_id"]), set(test_df["image_id"])]
assert not (ids[0] & ids[1]) and not (ids[0] & ids[2]) and not (ids[1] & ids[2]), "Image found in more than one split!"
assert sum(len(s) for s in ids) == len(df1), "Some images are missing from the splits!"
print("OK: no overlap between splits, total =", len(df1))


train -> 3581 images
val -> 765 images
test -> 768 images
OK: no overlap between splits, total = 5114
